In [1]:
import pandas as pd
import os
import numpy as np
import folium
import matplotlib.pyplot as plt
import textwrap

from matplotlib.backends.backend_pdf import PdfPages
from folium.plugins import FastMarkerCluster
from folium.plugins import MarkerCluster
from pathlib import Path
from datetime import datetime
from pyproj import Transformer

from reportlab.platypus import SimpleDocTemplate, Table, TableStyle
from reportlab.lib.pagesizes import A4
from reportlab.lib import colors

<div class="alert alert-info">
    <b>Erstellen einer .pdf zu allen gesammelten Datenbanken bzgl. ihres Rock Types</b><br>
    -> Implementierung mit einzelnen .csvs, bevor die Rock-Types über Notebook 1.2 ergänzt wurden
    -> Arbeit erstmal nur mit 1.1
</div>

In [2]:
# ---------- Pfad zu Ordner mit .csv Dateien -----------
base_dir = Path.cwd()

db_root = (
    base_dir.parent.parent 
    / "1_Acquisition" 
    / "1.1_Data-Acquisition-Wrapper"
    / "Angepasste_Datenbanken" / "Nach_Acquisition"
)

# ------- alle .csvs aus einem Ordner sammeln --------
csv_files = sorted(db_root.glob("*.csv"))

if not csv_files:
    raise FileNotFoundError(f"Keine .csv-Dateien in {db_root} gefunden.")

# --------- Zielfpad für .pdf ---------
output_pdf = base_dir / "Rock-Type_Analysis_without-added-rock-types.pdf"
print(output_pdf)

# ---------- Eine Seite pro csv ------------
with PdfPages(output_pdf) as pdf:
    for csv_path in csv_files:
        # -------- CSV einlesen ---------
        try:
            df = pd.read_csv(csv_path, low_memory=False)
        except Exception as e:
            # ---------- Fehlerbehandlung ------------
            fig = plt.figure(figsize=(8.27, 11.69))  
            fig.suptitle(f"Datei: {csv_path.name}", fontsize=14, fontweight="bold")
            fig.text(0.1, 0.8, f"Fehler beim Einlesen:\n{e}", fontsize=10)
            plt.axis("off")
            pdf.savefig(fig)
            plt.close(fig)
            continue

        total_rows = len(df)

        # --------- Prüfen ob Rock Type existiert ---------
        if "rock_type" in df.columns:
            rock_col = df["rock_type"]
            non_null = rock_col.notna().sum()
            percent = (non_null / total_rows * 100) if total_rows > 0 else 0.0

            # ---------- Einzigartige Werte -----------
            unique_vals = sorted(rock_col.dropna().astype(str).unique())
        else:
            non_null = 0
            percent = 0.0
            unique_vals = []

        # -------------- Text für Seite ---------------
        header_lines = [
            f"Datei: {csv_path.name}",
            "",
            f"Anzahl Zeilen insgesamt: {total_rows}",
            f"Zeilen mit Wert in 'rock_type': {non_null} ({percent:0.2f} %)",
            "",
            "Einzigartige Werte in 'rock_type':"
        ]

        # -------- Einzigartige Werte als Liste in Textform ----------
        if unique_vals:
            # ------- Kommatrennung für Zeichenkette
            unique_str = ", ".join(unique_vals)
        else:
            unique_str = "(Keine Werte oder Spalte 'rock_type' nicht vorhanden.)"

        # Zeilen umbrechen, damit es auf die Seite passt
        wrapped_unique_lines = textwrap.wrap(unique_str, width=100)

        # Neue Seite (Figure) erstellen
        fig = plt.figure(figsize=(8.27, 11.69))  # A4-Hochformat
        fig.suptitle("Rock-Type-Analyse", fontsize=14, fontweight="bold", y=0.97)

        # Alle Zeilen gesammelt ausgeben
        all_lines = header_lines + [""] + wrapped_unique_lines

        # Text von oben nach unten platzieren
        y_start = 0.9
        line_height = 0.025

        for i, line in enumerate(all_lines):
            y = y_start - i * line_height
            if y < 0.05:  # falls zu viele Zeilen: abbrechen
                fig.text(0.1, 0.05, "... (weitere Werte abgeschnitten)", fontsize=9)
                break
            fig.text(0.1, y, line, fontsize=10, va="top")

        plt.axis("off")
        pdf.savefig(fig)
        plt.close(fig)

print(f"PDF erstellt: {output_pdf}")


C:\Users\lucca\OneDrive\SPEICHER\Hochschule\7. Semester\Abschlussarbeit Bearbeitung\Jupyter Notebooks\2_Analysis\2.2_Rock-Type_Analysis\Rock-Type_Analysis_without-added-rock-types.pdf


PDF erstellt: C:\Users\lucca\OneDrive\SPEICHER\Hochschule\7. Semester\Abschlussarbeit Bearbeitung\Jupyter Notebooks\2_Analysis\2.2_Rock-Type_Analysis\Rock-Type_Analysis_without-added-rock-types.pdf


<div class="alert alert-info">
    <b>Erstellen einer .pdf zu allen gesammelten Datenbanken bzgl. ihres Rock Types</b><br>
    -> Implementierung mit bereits ergänzten Rock-Types über Notebook 1.2
    -> Überprüfung der daraus resultierenden .csv
</div>

In [3]:
# Basisverzeichnis (aktuelles Notebook-Verzeichnis)
base_dir = Path.cwd()

# Pfad zu "Komplette_Datenbank" 
db_root = (
    base_dir.parent.parent 
    / "1_Acquisition" 
    / "1.2_Rock-Type-Mapping"
    / "Rock-Type_Mapping"
)

# Alle Unterordner lesen, Ordner mit "neustem" Datum setzen
timestamp_folders = [f for f in db_root.iterdir() if f.is_dir()]
latest_folder = max(timestamp_folders, key=lambda f: f.stat().st_mtime)


# Komplette Datenbank als CSV
database_path = latest_folder / "finalized_database_with_lithology.csv"

print(database_path)

C:\Users\lucca\OneDrive\SPEICHER\Hochschule\7. Semester\Abschlussarbeit Bearbeitung\Jupyter Notebooks\1_Acquisition\1.2_Rock-Type-Mapping\Rock-Type_Mapping\2025-11-27_12-22-40\finalized_database_with_lithology.csv


In [4]:
# Ordner "Datenanalyse"
analysis_root = base_dir / "Rock-Type_Analysis"
analysis_root.mkdir(exist_ok=True)

output_pdf = base_dir / "Rock-Type_Analysis_with-added-rock-types.pdf"

print("Erzeugter Output-Ordner:")
print(output_pdf)

Erzeugter Output-Ordner:
C:\Users\lucca\OneDrive\SPEICHER\Hochschule\7. Semester\Abschlussarbeit Bearbeitung\Jupyter Notebooks\2_Analysis\2.2_Rock-Type_Analysis\Rock-Type_Analysis_with-added-rock-types.pdf


<div class="alert alert-info">
    <b>Erstellen einer .pdf zu allen gesammelten Datenbanken bzgl. ihres Rock Types</b><br>
    -> Wie viele Zeilen hat die .csv<br>
    -> In wie vielen Zeilen hat die Spalte "rock_type" Werte? (in %)<br>
    -> Welche Werte beinhaltet diese Spalte alles (Auflistung aller einzigartigen Attribute)
</div>

In [5]:
# ------- Gesamte Datenbank einmal einlesen --------
df = pd.read_csv(database_path, low_memory=False)

# ----- Alle vorhandenen Database_number-Werte aufsteigend sortiert --------
db_numbers = sorted(df["Database_number"].dropna().unique())


# ----- PDF mit einer Seite pro Database_number
with PdfPages(output_pdf) as pdf:
    for db_num in db_numbers:

        # -------- alle Zeilen dieser Datenbank holen ----------
        df_db = df[df["Database_number"] == db_num]
        total_rows = len(df_db)

        # -------- Database_name aus diesen Zeilen extrahieren --------
        db_name = df_db["Database_name"].dropna().astype(str).unique()
        db_name = db_name[0] if len(db_name) > 0 else "Unbekannt"

        # -------- Prüfen, ob Spalte 'rock_type' existiert -----------
        if "rock_type" in df_db.columns:
            rock_col = df_db["rock_type"]
            non_null = rock_col.notna().sum()
            percent = (non_null / total_rows * 100) if total_rows > 0 else 0.0

            unique_vals = sorted(rock_col.dropna().astype(str).unique())
        else:
            non_null = 0
            percent = 0.0
            unique_vals = []

        # ----------- Text für die Seite ------------
        header_lines = [
            f"Database_number: {db_num}",  # Kürzel
            f"Database_name:   {db_name}", # Kürzel
            "",
            f"Anzahl Zeilen insgesamt: {total_rows}",
            f"Zeilen mit Wert in 'rock_type': {non_null} ({percent:0.2f} %)",
            "",
            "Einzigartige Werte in 'rock_type':"
        ]

        if unique_vals:
            unique_str = ", ".join(unique_vals)
        else:
            unique_str = "(Keine Werte oder Spalte 'rock_type' nicht vorhanden.)"

        wrapped_unique_lines = textwrap.wrap(unique_str, width=100)

        # ------------ Neue Seite erstellen ------------
        fig = plt.figure(figsize=(8.27, 11.69)) 

        # ------------ Titel bestehend aus Database_number und Database_name -----------
        fig.suptitle(
            f"Rock-Type-Analyse – DB {db_num} ({db_name})",
            fontsize=14,
            fontweight="bold",
            y=0.97
        )

        all_lines = header_lines + [""] + wrapped_unique_lines

        y_start = 0.9
        line_height = 0.025

        for i, line in enumerate(all_lines):
            y = y_start - i * line_height
            if y < 0.05:
                fig.text(0.1, 0.05, "... (weitere Werte abgeschnitten)", fontsize=9)
                break
            fig.text(0.1, y, line, fontsize=10, va="top")

        plt.axis("off")
        pdf.savefig(fig)
        plt.close(fig)

print(f"PDF erstellt: {output_pdf}")

PDF erstellt: C:\Users\lucca\OneDrive\SPEICHER\Hochschule\7. Semester\Abschlussarbeit Bearbeitung\Jupyter Notebooks\2_Analysis\2.2_Rock-Type_Analysis\Rock-Type_Analysis_with-added-rock-types.pdf
